# Prepare environnement
## import package

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sb
import csv
import requests
from sklearn.preprocessing import StandardScaler #pip install scikit-learn

## Options

In [ ]:
pd.set_option('display.max_columns', None) # affiche toutes les colonnes
pd.set_option('display.max_colwidth', None)  # Affiche les colonnes dans toute leur largeur

## Create a dataframe

In [ ]:
data=pd.read_csv("fr.openfoodfacts.org.products.csv",sep="\t")# PB with csv delimiter

# Describe
## Structure

In [ ]:
data.columns

In [ ]:
data.info()

## Clean by col
###  Etape 1 : ne garder que les colonnes les plus renseignées
####  Oter les col 100% Nan

In [ ]:
#how many Nan value per column
data.isna().sum().sort_values()

In [ ]:
# need to replace 0 by nan
#data.replace("0.0",np.NaN)

In [ ]:
# % of Nan in each col
df_per_col_nan=(data.isnull().sum()/(len(data.iloc[:,:]))*100).round().sort_values()
df_per_col_nan

In [ ]:
# Show distribution of % of nan
plt.hist(df_per_col_nan)
plt.ylabel("Nombre de colonne", fontweight="bold")
plt.xlabel("% de Nan", fontweight="bold")
plt.grid(axis="y")
plt.title("Repartition du taux de Nan par colonne (162)",fontweight="bold", fontsize=15)

In [ ]:
# Remove col with more than 100% Nan
df_col_nan_100removed=data.dropna(axis=1,how="all")
df_col_nan_100removed.shape
#output : 146

#### Remove above 50% of Nan value (not significative)

In [ ]:
# Remove col with more than 50% Nan
df_50Nan_col=data.dropna(axis=1,thresh=int(0.5*len(data)))
df_50Nan_col.shape

In [ ]:
list_col_34=df_50Nan_col.columns
list_col_34

In [ ]:
df_50Nan_col.info()

In [ ]:
# Graph which show how col are informed : barplot
fig=plt.figure(figsize=(25,6))
for i in list_col_34:
    plt.bar(height=df_per_col_nan[i],x=i)
    ax= plt.gca()
    ax.set_xticklabels(labels=list_col_34,rotation= 45)
plt.title("Taux de Nan value pour chaque colonne conservées (34)", fontsize=15, fontweight="bold")
plt.xlabel('Colones',fontsize=12, fontweight="bold")
plt.xticks(fontsize=14)
plt.ylabel('% of Nan value',fontsize=12, fontweight="bold")

## Selection of usefull col (idee appli sur aliment energetique ou aliment sain)

In [ ]:
list_col_34

I will need to keep info relatives to:
- product identity : "code","product_name", ('brands'?), 'serving_size'
- ingredient/qualification: "ingredients_text"
- ingredient/composition : 'additives_n', 'additives', 'ingredients_from_palm_oil_n',
       'ingredients_that_may_be_from_palm_oil_n', 
- energetical :'energy_100g', 'fat_100g','saturated-fat_100g', 'carbohydrates_100g', 'sugars_100g', 'fiber_100g','proteins_100g', 
- quality : nutrition_grade_fr','nutrition-score-fr_100g','nutrition-score-uk_100g'

In [ ]:
# create list with usefull col
list_col_select18=["code","product_name", 'brands', 'serving_size',"ingredients_text",'additives_n', 'additives', 'ingredients_from_palm_oil_n',
"ingredients_that_may_be_from_palm_oil_n", 'energy_100g', 'fat_100g','saturated-fat_100g', 'carbohydrates_100g', 'sugars_100g', 'fiber_100g','proteins_100g', 
"nutrition_grade_fr","nutrition-score-fr_100g",'nutrition-score-uk_100g']
list_col_nrj=['code', 'product_name', 'brands', 'serving_size', 'nutrition_grade_fr',
       'energy_100g', 'fat_100g', 'saturated-fat_100g', 'carbohydrates_100g',
       'sugars_100g', 'fiber_100g', 'proteins_100g', 'nutrition-score-fr_100g',
       'nutrition-score-uk_100g']
len(list_col_nrj)

In [ ]:
# Create a df with those col
df_col_select18=df_50Nan_col.loc[:,df_50Nan_col.columns.isin(list_col_select18)]
df_col_nrj=df_50Nan_col.loc[:,df_50Nan_col.columns.isin(list_col_nrj)]
df_col_nrj.head()

In [ ]:
df_col_nrj.info()

In [ ]:
"""# Create a dataframe with Nan value
df_col_nrj_nan=pd.DataFrame(df_col_nrj.isnull().sum(),columns=[["Cat"],["Nan"]],index=True)
df_col_nrj_nan["Cat"]=list_col_nrj
df_col_nrj_nan"""
df2=pd.DataFrame(df_col_nrj.isnull().sum(),columns=["Number of Nan"])
df2.reset_index(inplace=True)
df2["% of Nan"]= ((df2["Number of Nan"]/len(df_col_nrj.iloc[:,:]))*100).round(1)

In [ ]:
# Distribution of all this col vs % of Nan by col : barplot
#create a list of col
fig=plt.figure(figsize=(25,6))
plt.bar(height=df2["% of Nan"],x=df2["index"])
ax= plt.gca()
ax.set_xticklabels(labels=df2["index"],rotation= 45)
plt.title("Taux de Nan value pour chaque catégorie relative à l'energie de l'aliment", fontsize=25, fontweight="bold")
plt.xlabel('Colones',fontsize=20, fontweight="bold")
plt.xticks(fontsize=14)
plt.ylabel('% of Nan value',fontsize=20, fontweight="bold")

## Look for aberrant values
### Look at distribution of each col

In [ ]:
# Create a list of col/variable by type:
df_col_nrj.dtypes
list_col_float14=df_col_nrj.select_dtypes('float').columns
list_col_string=df_col_nrj.select_dtypes('object').columns
list_col_string

In [ ]:
list_col_float14

#### Variable qualitative

In [ ]:
# check what there is inside:
df_string14=df_col_nrj.select_dtypes('object')
df_string14

In [ ]:
df_col_nrj=df_50Nan_col.loc[:,df_50Nan_col.columns.isin(list_col_nrj)]
df_col_nrj.head()

##### Colone code

In [ ]:
# Keep only row with no nan in column "code"(23):
#df_col_nrj3=df_col_nrj["code"].dropna() : to dangerous
df_col_nrj2= df_col_nrj[df_col_nrj['code'].notna()]
df_col_nrj3= df_col_nrj2[df_col_nrj2['product_name'].notna()]
len(df_col_nrj3)

In [ ]:
# Look at duplicated
len(df_col_nrj3["code"].unique())
# output: 302_883 (different de len (df):302_987, mean there are duplicates)

###### Deal with duplicated

In [ ]:
# Check which col code are duplicated :
pd.set_option('display.max_rows', None) 
duplicated2=df_col_nrj3[df_col_nrj3.duplicated(['code'], keep=False)].sort_values(by=['code'])
len(duplicated2)
# Output: 208 duplicated

**Output:** different case for same index:
###### **Same product**
**1/ similar in 3 columns (code, name and serving size):**
to differenciate them, we need to check if theses 3 columns('code',"serving_size","product_name") are the same:

In [ ]:
# Since sometimes same code number but different products
## I need to remove first the duplicated by comparing 2 col
dd=df_col_nrj3[df_col_nrj3.duplicated(['code',"serving_size","product_name"], keep=False)].sort_values(by=['code'])

In [ ]:
# Check col code duplicated
duplicate=df_col_nrj3[df_col_nrj3["code"].duplicated()]
print ("There are ",len(duplicate),"duplicates in code columns")

In [ ]:
# We can remove those:
# 1/ create a liste with those code
list_code_duplicated=dd["code"]
# Drop those duplicated
df_col_nrj3.drop_duplicates(subset=["code","serving_size","product_name"],inplace=True)

In [ ]:
len(df_col_nrj3)#output : 302 973 (90 removed)

In [ ]:
# Check duplicated
#df_col_nrj3[df_col_nrj3["code"].duplicated(keep=False)].sort_values("code") # check duplicated row

**2/ Similar but with missing important values:**
Remove rows if some duplicates have nan in important col (energy for 100g, serving size) 

In [ ]:
# Remove NaN 
df_col_nrj4= df_col_nrj3[df_col_nrj3['energy_100g'].notna()] # remove row if nan in energy col 
df_col_nrj5= df_col_nrj4[df_col_nrj4['serving_size'].notna()] # remove row if nan in serving size col

**3/ Same product if 2 columns are similar:**

In [ ]:
df_col_nrj5.drop_duplicates(subset=["code","energy_100g"],inplace=True) #remove duplicates if code and energy for 100g are the same
#df_col_nrj5[df_col_nrj5["code"].duplicated(keep=False)].sort_values("code") # check duplicated row

**4/ Cas ou erreur d'index:**
2 x le meme index, pas le meme nom de produit, pas meme valeur energetique.
On retrouve le doublon, avec pas le meme index, mais le reste identique
exemple : row 104660	et 161115

In [ ]:
# Remove if brands/ serving size and energy are similar
df_col_nrj5.drop_duplicates(subset=["brands","serving_size","energy_100g"],inplace=True) #remove duplicates if code and energy for 100g are the same
#df_col_nrj5[df_col_nrj5["code"].duplicated(keep=False)].sort_values("code") # check duplicated row

Try to remove bracket in col serving size

In [ ]:
#df_col_nrj5["serving_size"]=df_col_nrj5["serving_size"].map(lambda x: x.lstrip().rstrip("()"))
#it works, it remove only the last bracket


**5/ Cas ou nom possède des variations d'écriture de nom, size et que la valeur energetique n est pas la meme, mais proche**
cas de ["code"]: 78895300024 / 76186000028 / 9800892204
solution: moyenner les 2 valeurs, et supprimer le doublon

In [ ]:
dd5=df_col_nrj5[df_col_nrj5["code"].duplicated(keep=False)].sort_values("code") # check duplicated row
mean_nrj_double=dd5.groupby("code",as_index=False).mean()# measure la moyenne selon code
mean_nrj_double # need to exlude replace 0 by nan and explude nan

In [ ]:
# 1. Remove a la main with index row 
# 423 :Nutela: all others values are a bit different but close, keep mean value)
# 691 : noodle, all the others values are the same
df_col_nrj6=df_col_nrj5.drop([691,672,699])
df_col_nrj6[df_col_nrj6["code"].duplicated(keep=False)].sort_values("code") # check duplicated row

###### Cas ou meme index mais pas meme produit
cas code 16117  / 20048402	/ 853547003030 ??

In [ ]:
#Change index number for 2 cases
df_col_nrj6.iloc[5,0]=1016117	
df_col_nrj6.iloc[5,0]
df_col_nrj6[df_col_nrj6["code"]==1016117]

In [ ]:
#-Code= 20048402 or by index : 177550/214
df_col_nrj6.iloc[170,0]=20048402100
df_col_nrj6[df_col_nrj6["code"]==20048402100]

In [ ]:
# Change name code 853547003030 /index 104659
df_col_nrj6.iloc[108719,0]=853547003030100
df_col_nrj6.iloc[108718,0]=853547003030200
df_col_nrj6[df_col_nrj6["code"]==853547003030]#verifie single :ok

In [ ]:
list_col_float14

##### Replace by mean duplicated

In [ ]:
# replace by mean
test_replace=np.array(mean_nrj_double.loc[mean_nrj_double["code"]==9800892204,'energy_100g':'nutrition-score-uk_100g'])
df_col_nrj6.loc[df_col_nrj6["code"]==9800892204,'energy_100g':'nutrition-score-uk_100g']=test_replace[0]
df_col_nrj6.loc[df_col_nrj6["code"]==9800892204,:]

In [ ]:
# replace by mean: -	Code=35493012711 or by index : 29906	/ 237297
test_replace2=np.array(mean_nrj_double.loc[mean_nrj_double["code"]==35493012711,'energy_100g':'nutrition-score-uk_100g'])
df_col_nrj6.loc[df_col_nrj6["code"]==35493012711,'energy_100g':'nutrition-score-uk_100g']=test_replace2[0]
df_col_nrj6.loc[df_col_nrj6["code"]==35493012711,:]

In [ ]:
#Code=72220009941 or by index : 79578 / 670
test_replace3=np.array(mean_nrj_double.loc[mean_nrj_double["code"]==72220009941,'energy_100g':'nutrition-score-uk_100g'])
df_col_nrj6.loc[df_col_nrj6["code"]==72220009941,'energy_100g':'nutrition-score-uk_100g']=test_replace3[0]
df_col_nrj6.loc[df_col_nrj6["code"]==72220009941,:]

In [ ]:
#remove last duplicated
df_col_nrj6[df_col_nrj6["code"].duplicated(keep=False)].sort_values("code") # check duplicated row

In [ ]:
#Drop duplicated
df_col_nrj6.drop_duplicates(subset=["code"],inplace=True)
df_col_nrj6[df_col_nrj6["code"].duplicated(keep=False)].sort_values("code") # check duplicated row: ok

In [ ]:
len(df_col_nrj6["product_name"].unique())#output : 116 500
#df_col_nrj6=df_col_nrj6.drop(304) doesn t work

### Col product name

In [ ]:
df_col_nrj6.columns
df_col_nrj6.shape #output 155 399 row and 14 col

In [ ]:
len(df_col_nrj6['product_name'].unique())
#OUtput: 116 502, mean there are duplicated (DIFFERENT DE 155 399)

In [ ]:
# Remove all nan value from this key col
df_col_nrj6['product_name'].isnull().sum() #17762
#df_col2_clean_nrj=df_col_nrj.dropna(df_col_nrj["product_name"])

Remove duplicated from that col

In [ ]:
#check duplicated
#(df_col_nrj6[df_col_nrj6["product_name"].duplicated(keep=False)]).sort_values("product_name") # check duplicated row
#df_col_nrj6.drop_duplicates(subset=["brands","serving_size","energy_100g"]) #remove duplicates if code and energy for 100g are the same

#### Variable quantitative
##### Look at distribution

In [ ]:
# Check which col with float we have
list_col_float14

In [ ]:
# replace > 3700 by nan in column
df_col_nrj6["energy_100g"][df_col_nrj6["energy_100g"]> 3700] = np.nan
df_col_nrj6["energy_100g"].max() #output 3983

In [ ]:
# Deal with Nan Value: ignore them for graph
#filtered_data = df_col_nrj6[~np.isnan(df_col_nrj6)]
df_nrj100 = df_col_nrj6[(df_col_nrj6["energy_100g"] != 0)]
df_col_nrj6.info()

In [ ]:
df_col_nrj6.describe().round(2)

In [ ]:
df_col_nrj6[i].isna().count()

In [ ]:
# Create a dataframe with Nan value
df_nan2=pd.DataFrame(df_col_nrj6.isnull().sum(),columns=["Number of Nan"])
df_nan2.reset_index(inplace=True)
df_nan2["% of Nan"]= ((df_nan2["Number of Nan"]/len(df_col_nrj6))*100).round(1)
df_nan2

In [ ]:
list_col_nrj
df_col_nrj6.columns

In [ ]:
# recount nan in histogramme
# Graph which show how col are informed : barplot
fig=plt.figure(figsize=(25,6))
plt.bar(height=df_nan2["% of Nan"],x=list_col_nrj)
ax= plt.gca()
ax.set_xticklabels(labels=list_col_nrj,rotation= 45)
plt.title("Taux de Nan value pour chaque colonne conservées (14)post clean", fontsize=15, fontweight="bold")
plt.xlabel('Colones',fontsize=12, fontweight="bold")
plt.xticks(fontsize=14)
plt.ylabel('% of Nan value',fontsize=12, fontweight="bold")

In [ ]:
# histogramme pour une col<energy_100g>
fig=plt.figure(figsize=(20,10))
plt.title('fat_100g')
ax=plt.subplot(211)
df_col_nrj6["fat_100g"].dropna().hist(density=True)
sb.kdeplot(df_col_nrj6["fat_100g"].dropna())
plt.subplot(212, sharex=ax)
plt.boxplot(df_col_nrj6["fat_100g" ].dropna(),vert=False)
plt.xlim(0,100)

#clean extremes outliers

In [ ]:
# histogramme pour une col<energy_100g>
fig=plt.figure(figsize=(20,10))
ax=plt.subplot(211)
df_col_nrj6[df_col_nrj6["energy_100g"] < 3700]["energy_100g"].hist(density=True,bins=100)
sb.kdeplot(df_col_nrj6[df_col_nrj6["energy_100g"] < 3700]["energy_100g"])
plt.subplot(212, sharex=ax)
plt.boxplot(df_col_nrj6[df_col_nrj6["energy_100g"] < 3700]["energy_100g" ],vert=False)

In [ ]:
list_col_float14

In [ ]:
for i in list_col_float14 :
    df_col_nrj6.reset_index()
    fig=plt.figure(figsize=(20,10))
    if i == 'energy_100g':
        ax=plt.subplot(211)
        plt.title("Distribution de la variable "+i,fontsize=15, fontweight="bold")
        df_col_nrj6[df_col_nrj6["energy_100g"] < 5000]["energy_100g"].hist(density=True,bins=100)
        sb.kdeplot(df_col_nrj6[df_col_nrj6["energy_100g"] < 5000]["energy_100g"])
        plt.subplot(212, sharex=ax)
        plt.boxplot(df_col_nrj6[df_col_nrj6["energy_100g"] < 5000]["energy_100g" ],vert=False)
    else :
        ax=plt.subplot(211)
        plt.title("Distribution de la variable "+i,fontsize=15, fontweight="bold")
        df_col_nrj6[(df_col_nrj6[i] <= 100)&(df_col_nrj6[i]>0)][i].hist(density=True, bins=100)
        sb.kdeplot(df_col_nrj6[(df_col_nrj6[i] <= 100)&(df_col_nrj6[i]>0)][i])
        plt.subplot(212, sharex=ax)
        plt.boxplot(df_col_nrj6[(df_col_nrj6[i] <= 100)&(df_col_nrj6[i]>0)][i],vert=False)

In [ ]:
# remplacer toutes les valeurs oulier par nan > 100 for all and > 4000 for nrj (kj, au max 3700)
for i in list_col_float14:
    if i == 'energy_100g':
        df_col_nrj6[df_col_nrj6["energy_100g"] > 3700]= np.nan
    else :
        df_col_nrj6[(df_col_nrj6[i] > 100)&(df_col_nrj6[i]<=0)][i]= np.nan
df_col_nrj6.describe().round(2)

In [ ]:
#replace neg values and values above 100 by Nans
for i in list_col_float14[1:-1]:
    df_col_nrj6.loc[(df_col_nrj6[i]>100),i]= np.nan
    df_col_nrj6.loc[(df_col_nrj6[i]<0.0),i]= np.nan
df_col_nrj6.describe().round()

In [ ]:
list_nutri_grade=df_col_nrj6['nutrition_grade_fr'].unique()
#list_nutri_grade
nbline=[]
for grade in list_nutri_grade:
    nbline.append(len(df_col_nrj6[df_col_nrj6['nutrition_grade_fr']==grade]))
nlinetot=df_col_nrj6.shape[0]
nbline[2] = df_col_nrj6['nutrition_grade_fr'].isna().sum() #properly count nans
list_perc_grade=np.array(nbline)/nlinetot
sum(list_perc_grade)# verifie que sum=1

In [ ]:
# create a pie chart
fig = plt.figure(figsize =(10, 7))
plt.pie(list_perc_grade, labels = list_nutri_grade)

# Traiter les valeurs aberantes 2
## Remplacer les nan par o dans fiber

In [ ]:
df_col_nrj6["fiber_100g"]= df_col_nrj6["fiber_100g"].fillna(0)
df_col_nrj6["fiber_100g"].isnull().count()#output: OK 155395

## Remplacer les nan par moyenne, par categories d aliment ou par score?

In [ ]:
data["pnns_groups_1"].unique()

In [ ]:
data["pnns_groups_1"]
#1rst step, add this col in df in use
df_pnns_group1=data[["pnns_groups_1","pnns_groups_2"]] # create a df to merge it easily
df_col_nrj7=pd.merge(df_col_nrj6, df_pnns_group1, left_index=True, right_index=True, how = "left")
df_col_nrj7.info()

In [ ]:
df_col_nrj7["pnns_groups_2"].unique()

In [ ]:
#How to capitalize first character of string in Python
#df_col_nrj7["pnns_groups_2"].str.capitalize().unique()

In [ ]:
#df_col_nrj7.loc[:,["pnns_groups_1","pnns_groups_2"]]

In [ ]:
df_col_nrj7.to_csv(r'C:\Users\Julie\Desktop\PROJET-03\clean_df_col_nrj7.csv')

In [ ]:
df_col_nrj7.shape
df_col_nrj7.describe().round(2)